In [ ]:
DATA_DIR = '/content/drive/MyDrive/HVTC/'

In [ ]:
import pandas as pd, numpy as np, torch

TS = "Time step"

# ---------- Bước 1+2: dựng lại txs, wal từ file đã lưu ----------
tx_idx  = pd.read_csv(f"{DATA_DIR}/tx_index.csv")     # txId, Time step, tx_idx, split
tx_lab  = pd.read_csv(f"{DATA_DIR}/tx_labels.csv")     # tx_idx, y, label_mask, *_mask
txs = tx_idx.merge(tx_lab, on="tx_idx", how="left", validate="1:1").sort_values("tx_idx")

addr_idx = pd.read_csv(f"{DATA_DIR}/addr_index.csv")   # address, Time step, addr_idx, split, eval_ok
addr_lab = pd.read_csv(f"{DATA_DIR}/addr_labels.csv")  # addr_idx, y, label_mask, *_mask, val_strict, test_strict
wal = addr_idx.merge(addr_lab, on="addr_idx", how="left", validate="1:1").sort_values("addr_idx")

print("txs:", txs.shape, "| wal:", wal.shape)
assert (txs["tx_idx"].to_numpy() == np.arange(len(txs))).all()
assert (wal["addr_idx"].to_numpy() == np.arange(len(wal))).all()

# ---------- Bước 3: đặc trưng đã chuẩn hoá ----------
X_tx  = np.load(f"{DATA_DIR}/X_tx.npy")
X_wal = np.load(f"{DATA_DIR}/X_wal.npy")
print("X_tx:", X_tx.shape, "| X_wal:", X_wal.shape)

# ---------- Bước 4: 49 đồ thị HeteroData ----------
graphs = torch.load(f"{DATA_DIR}/graphs.pt", weights_only=False)
print("Số đồ thị:", len(graphs))

# ---------- Bước 6: trọng số lớp ----------
ckpt = torch.load(f"{DATA_DIR}/class_weights.pt")
cw_tx, pw_tx   = ckpt["cw_tx"], ckpt["pw_tx"]
cw_addr, pw_addr = ckpt["cw_addr"], ckpt["pw_addr"]
print("cw_tx:", cw_tx, "| cw_addr:", cw_addr)

In [ ]:
!pip install -q torch-geometric
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv
import pandas as pd

# Khai báo lại hằng số TS
TS = "Time step"


# wal = pd.read_csv(f"{DATA_DIR}/addr_labels.csv")

# ---- map (address, time_step) -> global address uid ----
addr_uid_map = {a: i for i, a in enumerate(wal["address"].unique())}
n_addr_uid = len(addr_uid_map)
wal["addr_uid"] = wal["address"].map(addr_uid_map)
addr_uid_per_step = {t: torch.from_numpy(wal[wal[TS] == t]["addr_uid"].to_numpy())
                    for t in range(1, 50)}

In [ ]:
class TGNLite(nn.Module):
    def __init__(self, metadata, d_hid, n_classes=2):
        super().__init__()
        self.conv1 = HeteroConv({et: SAGEConv((-1, -1), d_hid) for et in metadata[1]}, aggr="sum")
        self.gru = nn.GRUCell(d_hid, d_hid)
        self.lin_tx   = nn.Linear(d_hid, n_classes)
        self.lin_addr = nn.Linear(d_hid, n_classes)

    def spatial(self, g):
        x = self.conv1(g.x_dict, g.edge_index_dict)
        return {k: F.relu(v) for k, v in x.items()}

    def step(self, g, hidden_state):
        """hidden_state: tensor (n_addr_uid, d_hid), cập nhật in-place theo t."""
        x = self.spatial(g)
        logit_tx = self.lin_tx(x["tx"])

        uids = addr_uid_per_step[g.ts]
        h_prev = hidden_state[uids]
        h_new = self.gru(x["addr"], h_prev)
        hidden_state[uids] = h_new
        logit_addr = self.lin_addr(h_new)
        return logit_tx, logit_addr, hidden_state

In [ ]:
from torch_geometric.transforms import ToUndirected

to_undirected = ToUndirected()

def prep_graph(g):
    g = g.clone()
    del g["addr", "to", "addr"]     # bỏ addr->addr, đã quyết định ở Bước 4
    g = to_undirected(g)            # thêm cạnh ngược thành quan hệ riêng
    return g

prepped = {t: prep_graph(g) for t, g in graphs.items()}
print(prepped[1].edge_types)        # kiểm tra: có rev_, không có (addr,to,addr)

In [ ]:
D_HID = 64
def masked_loss(logits, y, mask, weight):
    per_node = F.cross_entropy(logits, y.clamp(min=0), reduction="none")
    w = mask.float() * weight
    return (per_node * w).sum() / w.sum().clamp(min=1e-8)

model = TGNLite(prepped[1].metadata(), D_HID)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)


In [ ]:
def full_pass(train, warmup_only=False):
    """Chạy tuần tự 1..49, tính loss chỉ ở step có mask tương ứng."""
    model.train() if train else model.eval()
    hidden = torch.zeros(n_addr_uid, D_HID)
    total_loss = 0.0
    for t in range(1, 50):
        g = prepped[t]
        w_tx   = cw_tx[g["tx"].y.clamp(min=0)]
        w_addr = cw_addr[g["addr"].y.clamp(min=0)]
        logit_tx, logit_addr, hidden = model.step(g, hidden)
        if train and g.split == "train":
            key = "train_mask"
        elif not train:
            key = "val_mask" if g.split == "val" else "test_mask" if g.split == "test" else None
        else:
            key = None
        if key is not None:
            loss = (masked_loss(logit_tx,   g["tx"].y,   g["tx"][key],   w_tx) +
                    masked_loss(logit_addr, g["addr"].y, g["addr"][key], w_addr))
            total_loss += loss
        hidden = hidden.detach() if not train else hidden   # giữ gradient trong 1 epoch train, cắt khi eval
    return total_loss

In [ ]:
# ---- addr_uid: ánh xạ address -> uid, dùng addr_idx đã lưu sẵn, không tính lại từ wal filter ----
addr_uid_map = {a: i for i, a in enumerate(wal["address"].unique())}
n_addr_uid = len(addr_uid_map)

# map addr_idx (toàn cục, 0..920690) -> addr_uid (theo address, không theo time step)
addr_idx_to_uid = wal["address"].map(addr_uid_map).to_numpy()   # độ dài = len(wal), thứ tự = addr_idx

addr_uid_per_step = {
    t: torch.from_numpy(addr_idx_to_uid[graphs[t]["addr"].n_id.numpy()])
    for t in graphs
}

# ---- kiểm tra bắt buộc trước khi train ----
for t in graphs:
    n_graph = graphs[t]["addr"].x.shape[0]
    n_uid   = len(addr_uid_per_step[t])
    assert n_graph == n_uid, f"step {t}: graph có {n_graph} node addr nhưng uid có {n_uid}"
print("Khớp toàn bộ 49 step.")

In [ ]:
from sklearn.metrics import f1_score

# ---- train: full BPTT qua cả 49 step mỗi epoch, chỉ backward loss ở step train ----
best_f1, best_state, no_improve = -1, None, 0
for epoch in range(1, 41):
    opt.zero_grad()
    loss = full_pass(train=True)
    loss.backward(); opt.step()

    if epoch % 5 == 0:
        with torch.no_grad():
            hidden = torch.zeros(n_addr_uid, D_HID)
            ys, ps = {"tx": [], "addr": []}, {"tx": [], "addr": []}
            for t in range(1, 50):
                g = prepped[t]
                lt, la, hidden = model.step(g, hidden)
                if g.split == "val":
                    for nt, logit in (("tx", lt), ("addr", la)):
                        m = g[nt]["val_mask"]
                        ys[nt].append(g[nt].y[m]); ps[nt].append(logit.argmax(1)[m])
            f1v = 0.5 * f1_score(torch.cat(ys["tx"]), torch.cat(ps["tx"]), pos_label=1, zero_division=0) + \
                 0.5 * f1_score(torch.cat(ys["addr"]), torch.cat(ps["addr"]), pos_label=1, zero_division=0)
        print(f"epoch {epoch}: loss={loss.item():.3f} val_F1={f1v:.4f}")
        if f1v > best_f1:
            best_f1, best_state, no_improve = f1v, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_improve += 1
            if no_improve >= 4: break

model.load_state_dict(best_state)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# ---- đánh giá test: chạy lại từ đầu (1..49) để hidden state đúng lịch sử ----
with torch.no_grad():
    hidden = torch.zeros(n_addr_uid, D_HID)
    ys, ps = {"tx": [], "addr": []}, {"tx": [], "addr": []}
    for t in range(1, 50):
        g = prepped[t]
        lt, la, hidden = model.step(g, hidden)
        if g.split == "test":
            for nt, logit in (("tx", lt), ("addr", la)):
                        m = g[nt]["test_mask"]
                        ys[nt].append(g[nt].y[m]); ps[nt].append(logit.argmax(1)[m])
    for nt in ("tx", "addr"):
        y, p = torch.cat(ys[nt]).numpy(), torch.cat(ps[nt]).numpy()
        print(f"[{nt}] TGN-lite test: P={precision_score(y,p,pos_label=1,zero_division=0):.3f} "
              f"R={recall_score(y,p,pos_label=1,zero_division=0):.3f} "
              f"F1={f1_score(y,p,pos_label=1,zero_division=0):.3f} (n={len(y)})")

ADGNN

In [ ]:
import numpy as np, torch

graphs = torch.load(f"{DATA_DIR}/graphs.pt", weights_only=False)   # dùng graphs GỐC (còn addr-addr), không dùng prepped

ALPHA, BETA, GAMMA, DELTA, T = 0.4, 0.2, 0.2, 0.2, 10    # cố định, không học, tổng = 1

def row_normalize(edge_index, n):
    src, dst = edge_index
    deg = torch.zeros(n).scatter_add_(0, dst, torch.ones(dst.numel()))
    w = 1.0 / deg[dst].clamp(min=1)
    return src, dst, w

def prop(x, src, dst, w, n):
    """Â @ x, dùng scatter (không cần dense adjacency)."""
    out = torch.zeros_like(x)
    return out.index_add_(0, dst, w.unsqueeze(-1) * x[src])

def adgnn_features(x, edge_index):
    n = x.shape[0]
    if edge_index.numel() == 0:
        z = torch.zeros_like(x)
        return torch.cat([x, z, z, z, x], dim=1)         # không cạnh -> phần lan truyền = 0

    src, dst, w = row_normalize(edge_index, n)
    Ax  = prop(x, src, dst, w, n)
    Lx  = x - Ax
    LAx = x - prop(Ax, src, dst, w, n)

    source = ALPHA * x + BETA * LAx + GAMMA * Lx          # Eq.7 phần source term
    h_star, term = source.clone(), source.clone()
    for _ in range(T):                                     # Eq.21 Neumann truncation
        term = DELTA * prop(term, src, dst, w, n)
        h_star = h_star + term

    return torch.cat([x, Lx, h_star], dim=1)               # H(0), 1 loại "local" (Lx), H*

# ---------- Áp dụng cho từng time step, ghép lại theo đúng thứ tự tx_idx / addr_idx ----------
n_tx_feat, n_addr_feat = X_tx.shape[1], X_wal.shape[1]
X_tx_adgnn  = np.zeros((len(txs), n_tx_feat * 3), dtype="float32")
X_wal_adgnn = np.zeros((len(wal), n_addr_feat * 3), dtype="float32")

for t, g in graphs.items():
    tx_n_id = g["tx"].n_id.numpy()
    feat = adgnn_features(g["tx"].x, g["tx", "to", "tx"].edge_index)
    X_tx_adgnn[tx_n_id] = feat.numpy()

    addr_n_id = g["addr"].n_id.numpy()
    feat = adgnn_features(g["addr"].x, g["addr", "to", "addr"].edge_index)
    X_wal_adgnn[addr_n_id] = feat.numpy()

print(X_tx_adgnn.shape, X_wal_adgnn.shape)   # (203769, 546) và (920691, 165)
np.save("X_tx_adgnn.npy", X_tx_adgnn)
np.save("X_wal_adgnn.npy", X_wal_adgnn)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

def run_rf(X, df, name, class_weight="balanced_subsample"):
    y = df["y"].to_numpy()
    tr = df["train_mask"].to_numpy()
    rf = RandomForestClassifier(n_estimators=50, class_weight=class_weight,
                                random_state=42, n_jobs=-1)
    rf.fit(X[tr], y[tr])
    for split in ("val", "test"):
        m = df[f"{split}_mask"].to_numpy()
        p = rf.predict(X[m]); t = y[m]
        P = precision_score(t, p, pos_label=1, zero_division=0)
        R = recall_score(t, p, pos_label=1, zero_division=0)
        F1 = f1_score(t, p, pos_label=1, zero_division=0)
        print(f"[{name}] {split}: P={P:.3f} R={R:.3f} F1={F1:.3f} (n={m.sum()})")
    return rf

print("=== RF trên đặc trưng gốc (mốc so sánh) ===")
run_rf(X_tx, txs, "tx-gốc")
run_rf(X_wal, wal, "addr-gốc")

print("\n=== RF trên đặc trưng gốc + ADGNN-feature ===")
X_tx_combo  = np.hstack([X_tx, X_tx_adgnn])
X_wal_combo = np.hstack([X_wal, X_wal_adgnn])
run_rf(X_tx_combo, txs, "tx-ADGNN")
run_rf(X_wal_combo, wal, "addr-ADGNN")

=== RF trên đặc trưng gốc (mốc so sánh) ===
[tx-gốc] val: P=0.991 R=0.924 F1=0.956 (n=3513)
[tx-gốc] test: P=0.986 R=0.633 F1=0.771 (n=16670)

=== RF trên đặc trưng gốc + ADGNN-feature ===
[tx-ADGNN] val: P=0.987 R=0.932 F1=0.959 (n=3513)
[tx-ADGNN] test: P=0.989 R=0.650 F1=0.784 (n=16670)